Verificar que exista el access log light para empezar a probar el parseo y normalización.

In [1]:
!ls -lh /content

total 8.0K
drwx------ 5 root root 4.0K Jan  7 23:31 drive
drwxr-xr-x 1 root root 4.0K Dec 11 14:34 sample_data


Imports base estándar

In [2]:
import re
import pandas as pd
import numpy as np

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

Definir el **regex** del log light.

In [4]:
""" Esta regex se va a usar para tests y prod, así que hay que verificar bien que
lea bien todos los logs de la misma manera."""

log_pattern = re.compile(
    r'(?P<ip>\S+) - \S+ \[(?P<time>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<path>\S+) (?P<protocol>[^"]+)" '
    r'(?P<status>\d{3}) (?P<bytes>\d+) '
    r'"(?P<referer>[^"]*)" '
    r'"(?P<user_agent>[^"]*)"'
)

Carga y parseo del archivo , **mínimo funcional**, ver si puede hacerse más robusto...

In [5]:
rows = []

#log_file_ligth = "/content/access_040126_2200_light.txt" # Ajustar nombre si se va a usar otro.
log_file_ligth = "/content/drive/MyDrive/Proyectos Personales/NGIANX/LogsHeavyTest/access_070126_0600_heavy.txt"
with open(log_file_ligth, "r", encoding="utf-8", errors="ignore") as f:
  for line in f:
    match = log_pattern.search(line)
    if match:
      rows.append(match.groupdict())

df = pd.DataFrame(rows)
df.head()

,ip,time,method,path,protocol,status,bytes,referer,user_agent
0,187.175.231.142,07/Jan/2026:08:00:02 +0000,GET,/es,HTTP/1.1,200,6669,-,Zabbix
1,38.50.216.152,07/Jan/2026:08:00:06 +0000,GET,/api/v1/flights/flight-info/,HTTP/1.1,200,67877,https://aitulum.com/en/airlines/flight,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...
2,187.155.211.83,07/Jan/2026:08:00:06 +0000,GET,/api/v1/flights/flight-info/,HTTP/1.1,200,67877,https://aitulum.com/es/airlines/flight,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
3,187.175.231.142,07/Jan/2026:08:00:10 +0000,GET,/api/v1/flights/flight-info/,HTTP/1.1,200,67877,https://aitulum.com/es/airlines/flight,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
4,187.175.231.142,07/Jan/2026:08:00:12 +0000,GET,/api/v1/flights/flight-info/,HTTP/1.1,200,67877,https://aitulum.com/es,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...


Se inicia con la limpieza básica.

In [6]:
# Conversión de fecha.
df["time"] = pd.to_datetime(
    df["time"], format="%d/%b/%Y:%H:%M:%S %z", errors="coerce"
)

In [7]:
# Tipos numéricos.
df["status"] = df["status"].astype("int")
df["bytes"] = df["bytes"].astype("int")

Sanity Check (**sanitizar**).

In [8]:
df.info()

df.isna().sum()

df["status"].value_counts().head()

df["method"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4929 entries, 0 to 4928
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   ip          4929 non-null   object             
 1   time        4929 non-null   datetime64[ns, UTC]
 2   method      4929 non-null   object             
 3   path        4929 non-null   object             
 4   protocol    4929 non-null   object             
 5   status      4929 non-null   int64              
 6   bytes       4929 non-null   int64              
 7   referer     4929 non-null   object             
 8   user_agent  4929 non-null   object             
dtypes: datetime64[ns, UTC](1), int64(2), object(6)
memory usage: 346.7+ KB


,count
method,
GET,4268
HEAD,619
POST,39
PRI,2
\x16\x03\x01\x00\xEE\x01\x00\x00\xEA\x03\x03\x9EM2a]1\x8F+;\xAAs\xD9\xF1,1


Se guardará el **dataset** estructurado, va a ser:


1.   Dataset base
2.   Reutilizable
3.   Input para ML



In [9]:
df.to_csv("/content/drive/MyDrive/Proyectos Personales/NGIANX/output/Primer Notebook salida/Logs heavy/logs_structured.csv", index=False)
df.to_parquet("/content/drive/MyDrive/Proyectos Personales/NGIANX/output/Primer Notebook salida/Logs heavy/logs_structured.parquet")
# Se usa Parquet porque es ideal para modelos y grandes volúmenes. (80K logs al día aprox.)